# Lab 4 Parte 1 — Solución CNN grietas (solo docente)

Referencia con hiperparámetros recomendados y respuestas teóricas sugeridas.
Los alumnos trabajan solo con `cnn_grietas_estructuras_alumno_ia.ipynb` + bitácora de prompts.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_cnn, verificar_exploracion, verificar_dataloaders,
    verificar_arquitectura, verificar_entrenamiento, verificar_curvas,
    verificar_metricas, verificar_casos_locales, resumen_seccion,
)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import Resize, ToTensor, Normalize
from PIL import Image

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Entorno listo | device={device}")


## Contexto del dataset (Mendeley — grietas en hormigón)

| Clase | En obra significa… | Imágenes (subset) |
|-------|-------------------|-------------------|
| **Negative** | Superficie sin grieta visible | 800 train + 200 val |
| **Positive** | Superficie con grieta | 800 train + 200 val |

Imágenes **227×227** RGB; en el lab se redimensionan con `IMAGE_SIZE` para entrenar en CPU.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama CNN en inspección estructural

Las **CNN** aprenden filtros espaciales (bordes, texturas, grietas) a partir de píxeles.

### 📘 Subpreguntas
- **1.a** ¿Qué ventaja tiene una CNN frente a features manuales para grietas?
- **1.b** ¿Qué papel cumplen convolución y pooling?
- **1.c** ¿Por qué necesitamos miles de imágenes etiquetadas?


#### ✅ Respuesta sugerida

**1.a** La CNN aprende patrones locales (orientación de grieta, contraste) sin diseñar descriptores a mano.
**1.b** Convolución detecta patrones; pooling reduce dimensión y aporta invariancia a pequeños desplazamientos.
**1.c** Sin ejemplos variados (luz, textura) el modelo no generaliza a otras obras.


In [ ]:
COMPONENTES_CNN = ["convolución", "pooling", "ReLU", "flatten", "fully connected"]
print("Componentes CNN del laboratorio:")
for c in COMPONENTES_CNN:
    print(f"  · {c}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `COMPONENTES_CNN`
r = []
try:
    r = verificar_panorama_cnn(COMPONENTES_CNN)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama CNN', r)


## Pregunta 2 — Exploración del dataset

### 📘 Subpreguntas
- **2.a** ¿Están balanceadas las clases en train?
- **2.b** ¿Qué variabilidad visual esperas entre Positive y Negative?


#### ✅ Reflexión 2

Clases balanceadas; variabilidad de textura e iluminación entre fotos METU.


In [ ]:
# --- PRE-ESCRITO: rutas y conteos ---
RUTA_DATOS = Path("data/cracks_subset")
if not RUTA_DATOS.is_dir():
    raise FileNotFoundError("Ejecuta primero: python _preparar_datos.py o bash labs/setup.sh")

conteos = {}
for split in ("train", "val"):
    conteos[split] = {}
    for cls in ("Negative", "Positive"):
        d = RUTA_DATOS / split / cls
        conteos[split][cls] = len(list(d.glob("*.jpg")))

class_names = ["Negative", "Positive"]
display(pd.DataFrame(conteos))


In [ ]:
N_EJEMPLOS_MOSAICO = 4
fig, axes = plt.subplots(2, N_EJEMPLOS_MOSAICO, figsize=(2 * N_EJEMPLOS_MOSAICO, 4))
for row, cls in enumerate(["Negative", "Positive"]):
    paths = sorted((RUTA_DATOS / 'train' / cls).glob('*.jpg'))[:N_EJEMPLOS_MOSAICO]
    for col, p in enumerate(paths):
        axes[row, col].imshow(Image.open(p))
        axes[row, col].set_title(cls if col == 0 else '')
        axes[row, col].axis('off')
plt.suptitle('Muestras de entrenamiento')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `N_EJEMPLOS_MOSAICO`, `conteos`
r = []
try:
    r = verificar_exploracion(conteos, N_EJEMPLOS_MOSAICO)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Exploración', r)


## Pregunta 3 — Transformaciones y DataLoaders

### 📘 Subpreguntas
- **3.a** ¿Por qué normalizamos con media 0.5?
- **3.b** ¿Por qué `shuffle=True` solo en train?


#### ✅ Reflexión 3

Normalize centra píxeles; shuffle evita orden de archivos en train.


In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32
transform = transforms.Compose([
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
train_ds = datasets.ImageFolder(RUTA_DATOS / 'train', transform=transform)
val_ds = datasets.ImageFolder(RUTA_DATOS / 'val', transform=transform)
class_names = train_ds.classes
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Clases: {class_names} | train={len(train_ds)} | val={len(val_ds)}")


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `IMAGE_SIZE`, `BATCH_SIZE`, `train_loader`, `val_loader`
r = []
try:
    r = verificar_dataloaders(IMAGE_SIZE, BATCH_SIZE, train_loader, val_loader)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — DataLoaders', r)


## Pregunta 4 — Arquitectura CNN

### 📘 Subpreguntas
- **4.a** ¿Qué hace cada bloque Conv+ReLU+Pool?
- **4.b** ¿Por qué dropout antes de la salida?


#### ✅ Reflexión 4

Dos bloques conv capturan bordes y texturas; dropout reduce sobreajuste.


In [ ]:
N_FILTERS = 32
DROPOUT = 0.3

class CrackCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, N_FILTERS, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(N_FILTERS, N_FILTERS * 2, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(N_FILTERS * 2 * 16, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

modelo = CrackCNN().to(device)
print(modelo)


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `N_FILTERS`, `DROPOUT`
r = []
try:
    r = verificar_arquitectura(modelo, N_FILTERS, DROPOUT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Arquitectura', r)


## Pregunta 5 — Entrenamiento

### 📘 Subpreguntas
- **5.a** ¿Qué optimizador y loss usas para clasificación multiclase?
- **5.b** ¿Cómo detectas overfitting con train vs val?


#### ✅ Reflexión 5

CrossEntropy + Adam estándar; gap train-val indica overfitting.


In [ ]:
# --- PRE-ESCRITO: funciones de entrenamiento y evaluación ---
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")


In [ ]:
N_EPOCHS = 5
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
for epoch in range(N_EPOCHS):
    tl, ta = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
    vl, va = eval_epoch(modelo, val_loader, criterion, device)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    print(f"Época {epoch+1}/{N_EPOCHS} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}")
print("✅ Entrenamiento completado.")


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`, `LEARNING_RATE`
r = []
try:
    r = verificar_entrenamiento(history, N_EPOCHS, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Entrenamiento', r)


## Pregunta 6 — Curvas de entrenamiento

### 📘 Subpreguntas
- **6.a** ¿Val loss sube mientras train baja?
- **6.b** ¿Cuántas épocas serían suficientes en producción?


#### ✅ Reflexión 6

Si val loss sube, early stopping o más regularización.


In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'], label='val')
ax1.set_title('Pérdida'); ax1.legend()
ax2.plot(epochs, history['train_acc'], label='train')
ax2.plot(epochs, history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`
r = []
try:
    r = verificar_curvas(history, N_EPOCHS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Curvas', r)


## Pregunta 7 — Métricas en validación

### 📘 Subpreguntas
- **7.a** ¿Qué clase se confunde más?
- **7.b** ¿Un falso positivo (grieta) es más grave que un falso negativo?


#### ✅ Reflexión 7

Revisar recall de Positive (grieta) para alertas de seguridad.


In [ ]:
modelo.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = modelo(images).argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())
acc_val = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión (val)')
plt.tight_layout()
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))
print(f"Accuracy validación: {acc_val:.3f}")


In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `acc_val`, `cm`
r = []
try:
    r = verificar_metricas(acc_val, cm)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Métricas', r)


## Pregunta 8 — Casos locales

### 📘 Subpreguntas
- **8.a** ¿La CNN mira la grieta o artefactos (sombra, mancha)?
- **8.b** ¿Qué harías para auditar un falso positivo?


#### ✅ Reflexión 8

Casos locales revelan si el modelo usa contexto o artefactos.


In [ ]:
N_CASOS_MOSTRADOS = 3
modelo.eval()
fig, axes = plt.subplots(1, N_CASOS_MOSTRADOS, figsize=(3 * N_CASOS_MOSTRADOS, 3))
if N_CASOS_MOSTRADOS == 1:
    axes = [axes]
for i in range(N_CASOS_MOSTRADOS):
    img, label = val_ds[i]
    with torch.no_grad():
        pred = modelo(img.unsqueeze(0).to(device)).argmax(1).item()
    vis = img.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    axes[i].imshow(vis.clip(0, 1))
    axes[i].set_title(f"real: {class_names[label]} | pred: {class_names[pred]}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `N_CASOS_MOSTRADOS`
r = []
try:
    r = verificar_casos_locales(N_CASOS_MOSTRADOS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Casos locales', r)


## Pregunta 9 — Reflexión ingeniería

### 📘 Subpreguntas
- **9.a** ¿Dónde desplegarías este modelo en un puente o edificio?
- **9.b** ¿Qué datos adicionales recolectarías en obra?
- **9.c** ¿La CNN sustituye la inspección normativa?


#### ✅ Respuesta sugerida

- **Triaje** de fotos de dron o cámaras fijas; no sustituye perito ni NDT.
- **Falsos positivos** por manchas/humedad; **falsos negativos** en grietas finas.
- Validar con inspección visual, repetir bajo distinta iluminación y documentar en informe.


## Cierre docente

La CNN aprende patrones de píxeles; en obra combínala con protocolos de inspección y datos del entorno.
